# RESIN: HolisQA loader and hop-wise evidence buffer

This notebook accompanies [Meng, Parekh, and Han, arXiv:2610.26782](https://arxiv.org/abs/2610.26782). It streams two public HolisQA examples and gives a tiny API-free illustration of typed retrieval, provenance recovery, and delayed composition.

In [ ]:
import json
import re
import urllib.request
from pprint import pprint

## 1. Stream two public HolisQA records

HolisQA is released in five scientific domains. Reading two lines from the biology JSONL avoids downloading the complete benchmark.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/Pwnb/CausalRAG2/main/data/holisqa/biology/qa.jsonl'
with urllib.request.urlopen(DATA_URL) as response:
    examples = [json.loads(response.readline()) for _ in range(2)]

for row in examples:
    compact = {key: row[key] for key in ('qa_id', 'category', 'question', 'answer', 'context_sentence_ids')}
    print(json.dumps(compact, indent=2, ensure_ascii=False))

## 2. Build a tiny source-linked reasoning-unit store

The normalized units are searchable addresses. Their `source_id` fields retain the original wording that will be used for the final decision.

In [ ]:
passages = {
    'P1': 'The bus costs $4. Reaching it requires a 10-minute walk, followed by a 35-minute ride.',
    'P2': 'The metro costs $8. Reaching it requires a 5-minute walk, followed by a 25-minute ride.',
    'P3': 'A bicycle rental costs $6 per hour and is available beside the station.',
}

units = [
    {'id': 'U1', 'kind': 'fare_fact',   'entity': 'bus',   'text': 'bus fare costs 4 dollars', 'source_id': 'P1'},
    {'id': 'U2', 'kind': 'access_fact', 'entity': 'bus',   'text': 'bus requires 10 minute walk', 'source_id': 'P1'},
    {'id': 'U3', 'kind': 'travel_fact', 'entity': 'bus',   'text': 'bus ride takes 35 minutes', 'source_id': 'P1'},
    {'id': 'U4', 'kind': 'fare_fact',   'entity': 'metro', 'text': 'metro fare costs 8 dollars', 'source_id': 'P2'},
    {'id': 'U5', 'kind': 'access_fact', 'entity': 'metro', 'text': 'metro requires 5 minute walk', 'source_id': 'P2'},
    {'id': 'U6', 'kind': 'travel_fact', 'entity': 'metro', 'text': 'metro ride takes 25 minutes', 'source_id': 'P2'},
    {'id': 'U7', 'kind': 'fare_fact',   'entity': 'bicycle', 'text': 'bicycle rental costs 6 dollars', 'source_id': 'P3'},
]
pprint(units)

## 3. Plan one-hop needs and fill the buffer

Each route receives its own retrieval budget. No hop decides the final answer, and all selected units remain available until composition.

In [ ]:
question = 'Which route reaches work within 40 minutes and costs at most $10: bus or metro?'
plan = [
    {'hop_id': 'H1', 'need': 'bus fare walk time ride time', 'target': 'bus'},
    {'hop_id': 'H2', 'need': 'metro fare walk time ride time', 'target': 'metro'},
]

def tokens(text):
    return set(re.findall(r'[a-z0-9]+', text.casefold()))

def retrieve(hop, top_k=3):
    need_tokens = tokens(hop['need'])
    scored = []
    for unit in units:
        overlap = len(need_tokens & tokens(unit['text']))
        target_bonus = 3 if unit['entity'] == hop['target'] else 0
        scored.append({**unit, 'score': overlap + target_bonus})
    return sorted(scored, key=lambda row: (-row['score'], row['id']))[:top_k]

reasoning_buffer = {hop['hop_id']: {'plan': hop, 'candidates': retrieve(hop)} for hop in plan}
pprint(reasoning_buffer)

## 4. Recover source evidence and compose once

The final step follows provenance pointers and reads the original passages. This deterministic parser stands in for RESIN's language-model composer.

In [ ]:
source_ids = {
    candidate['source_id']
    for hop in reasoning_buffer.values()
    for candidate in hop['candidates']
}
selected_passages = {source_id: passages[source_id] for source_id in sorted(source_ids)}
print('Recovered evidence:')
pprint(selected_passages)

def route_values(route, passage):
    pattern = rf'{route} costs \$(\d+).*?(\d+)-minute walk.*?(\d+)-minute ride'
    fare, walk, ride = map(int, re.search(pattern, passage, flags=re.I).groups())
    return {'fare': fare, 'walk': walk, 'ride': ride, 'total_minutes': walk + ride}

comparison = {
    'bus': route_values('bus', selected_passages['P1']),
    'metro': route_values('metro', selected_passages['P2']),
}
eligible = [route for route, values in comparison.items() if values['total_minutes'] <= 40 and values['fare'] <= 10]

print('\nComparison:')
pprint(comparison)
print('Answer:', eligible[0] if len(eligible) == 1 else eligible)

### Scope

This notebook captures RESIN's division of labor: plan focused needs, retrieve source-linked alternatives, and make the decision only from the assembled original evidence. It does not reproduce the paper's LLM-based extraction, embedding retrieval, generation, or RAGAS evaluation.